# Qwen2-VL-2B x full MMAD - Kaggle T4
Runs all 39,670 questions over 8,366 images. Data is processed one source archive at a time to stay within Kaggle disk limits. All outputs are saved under `/kaggle/working`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.49,<5', 'accelerate>=1.2', 'qwen-vl-utils>=0.0.8', 'remotezip>=0.12', 'pillow', 'requests', 'pandas', 'matplotlib', 'seaborn'], check=True)

In [ ]:
from pathlib import Path
import os, subprocess, sys
WORK = Path('/kaggle/working')
REPO = WORK / 'mini-world-model'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/anhsown/mini-world-model', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
BENCH = REPO / 'research/mmad_model_benchmark'
DATA = WORK / 'mmad_full_data'
OUT = WORK / 'qwen2_vl_mmad_full'
assert BENCH.exists()
os.chdir(BENCH)
sys.path.insert(0, str(BENCH))
print('benchmark:', BENCH)
print('persistent session output:', OUT)

In [ ]:
# Build metadata only: 39,670 questions, no images downloaded yet.
subprocess.run([sys.executable, 'prepare_full.py', '--output', str(DATA), '--metadata-only'], check=True)
import json
manifest = json.loads((DATA/'full_manifest.json').read_text(encoding='utf-8'))
assert len(manifest['records']) == 39670 and manifest['unique_images'] == 8366
print('questions:', len(manifest['records']), 'images:', manifest['unique_images'])

In [ ]:
# Process one source archive at a time. Safe to rerun: completed sample IDs are skipped.
import shutil
ARCHIVES = ['DS-MVTec', 'MVTec-AD', 'MVTec-LOCO', 'VisA', 'GoodsAD']
for archive_name in ARCHIVES:
    print('\n' + '='*80 + f'\nARCHIVE: {archive_name}\n' + '='*80)
    subprocess.run([sys.executable, 'prepare_full.py', '--output', str(DATA), '--archives', archive_name, '--range-download'], check=True)
    subprocess.run([sys.executable, 'models/qwen2_vl/run_full.py', '--data', str(DATA), '--output', str(OUT), '--archives', archive_name, '--batch-size', '4', '--checkpoint-every', '100'], check=True)
    shutil.make_archive(str(WORK/'qwen2_vl_mmad_full_partial'), 'zip', OUT)
    shutil.rmtree(DATA/'images', ignore_errors=True)
    print('checkpoint:', WORK/'qwen2_vl_mmad_full_partial.zip')

In [ ]:
# Recompute final metrics against the complete manifest.
subprocess.run([sys.executable, 'evaluate.py', '--manifest', str(DATA/'full_manifest.json'), '--predictions', str(OUT/'predictions.jsonl'), '--output', str(OUT)], check=True)

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
metrics = json.loads((OUT/'metrics.json').read_text(encoding='utf-8'))
scored = pd.read_csv(OUT/'predictions_scored.csv')
display(pd.DataFrame(metrics['per_task']).T)
display(pd.DataFrame(metrics['per_source']).T)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
task = pd.DataFrame(metrics['per_task']).T.reset_index(names='task')
sns.barplot(data=task, x='accuracy', y='task', ax=axes[0], color='#4C78A8')
axes[0].set_xlim(0, 1); axes[0].set_title('Accuracy by MMAD task')
source = pd.DataFrame(metrics['per_source']).T.reset_index(names='source')
sns.barplot(data=source, x='accuracy', y='source', ax=axes[1], color='#59A14F')
axes[1].set_xlim(0, 1); axes[1].set_title('Accuracy by source dataset')
cm = pd.crosstab(scored['truth'], scored['prediction'], dropna=False)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title('Answer confusion matrix')
plt.tight_layout()
plt.savefig(OUT/'full_mmad_analysis.png', dpi=180, bbox_inches='tight')
plt.show()
print(json.dumps(metrics, indent=2))

In [ ]:
import shutil, hashlib
archive = shutil.make_archive(str(WORK/'qwen2_vl_mmad_full_results'), 'zip', OUT)
digest = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
print('DOWNLOAD:', archive)
print('SHA256:', digest)